<a href="https://colab.research.google.com/github/Arma3071/Flyrank-AI-notebooks/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# FlyRank Internship - Week 3 Data Contract
# ============================================================

# Install dependencies
!pip -q install duckdb pandas scikit-learn pyarrow

import duckdb
import pandas as pd

# ------------------------------------------------------------
# Connect to Hugging Face
# ------------------------------------------------------------

HF_TOKEN = ""

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLE = f"""
read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
"""

print("Connected!")

Connected!


DATA CONTRACT

1. One row:
   One row represents one content item for one client on one report_date.

2. Tables:
   fact_content_daily_performance

3. Time Window:
   month = '2026-03'

4. Prediction:
   Predict whether a page receives high organic traffic
   (proxy = above-median GSC clicks).

5. Excluded:
   Future information and any metrics unavailable
   at prediction time.

In [2]:
grain = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows
FROM {TABLE}
WHERE strftime(report_date,'%Y-%m')='2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*)>1
LIMIT 10
""").df()

print(grain)

if len(grain)==0:
    print("PASS: Grain verified (one row per client/content/date).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, rows]
Index: []
PASS: Grain verified (one row per client/content/date).


In [3]:
summary = con.sql(f"""
SELECT

COUNT(*) AS total_rows,

MIN(report_date) AS first_day,

MAX(report_date) AS last_day

FROM {TABLE}

WHERE strftime(report_date,'%Y-%m')='2026-03'
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


In [4]:
availability = con.sql(f"""
SELECT

COUNT(*) AS available_rows

FROM {TABLE}

WHERE

strftime(report_date,'%Y-%m')='2026-03'

AND gsc_data_available IS TRUE

AND ga4_data_available IS TRUE
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,364347


In [5]:
features = con.sql(f"""
SELECT

report_date,

client_hash_id,

content_hash_id,

gsc_impressions,

gsc_avg_position,

ga4_sessions,

ga4_users,

scroll_events,

gsc_clicks

FROM {TABLE}

WHERE

strftime(report_date,'%Y-%m')='2026-03'

AND gsc_data_available IS TRUE

AND ga4_data_available IS TRUE

LIMIT 50000
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_users,scroll_events,gsc_clicks
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,5.400000,1,1,0,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,5.666667,2,1,0,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,5.156425,2,2,0,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,7.694444,1,1,0,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,6.167885,1,1,0,1


In [6]:
availability_notes = pd.DataFrame({
    "Feature":[
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_users",
        "scroll_events"
    ],
    "Available when?":[
        "Known from historical Search Console data.",
        "Computed from historical ranking information.",
        "Known from historical GA4 sessions.",
        "Known from historical GA4 users.",
        "Recorded before prediction time."
    ]
})

availability_notes

,Feature,Available when?
0,gsc_impressions,Known from historical Search Console data.
1,gsc_avg_position,Computed from historical ranking information.
2,ga4_sessions,Known from historical GA4 sessions.
3,ga4_users,Known from historical GA4 users.
4,scroll_events,Recorded before prediction time.


In [7]:
median_clicks = features.gsc_clicks.median()

features["label"] = (
    features.gsc_clicks >= median_clicks
).astype(int)

features.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_users,scroll_events,gsc_clicks,label
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,5.400000,1,1,0,0,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,5.666667,2,1,0,0,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,5.156425,2,2,0,0,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,7.694444,1,1,0,0,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,6.167885,1,1,0,1,1


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = features[
[
"gsc_impressions",
"gsc_avg_position",
"ga4_sessions",
"ga4_users",
"scroll_events"
]
]

y = features["label"]

X_train,X_test,y_train,y_test=train_test_split(
X,
y,
test_size=0.2,
random_state=42
)

model=RandomForestClassifier(
n_estimators=100,
random_state=42
)

model.fit(X_train,y_train)

pred=model.predict(X_test)

honest_accuracy=accuracy_score(y_test,pred)

print("Honest Accuracy:",honest_accuracy)

Honest Accuracy: 0.6494


In [9]:
X_leak = X.copy()

# DELIBERATE LEAKAGE
X_leak["future_label"] = y

X_train,X_test,y_train,y_test=train_test_split(
X_leak,
y,
test_size=0.2,
random_state=42
)

model=RandomForestClassifier(
n_estimators=100,
random_state=42
)

model.fit(X_train,y_train)

pred=model.predict(X_test)

leak_accuracy=accuracy_score(y_test,pred)

print("Accuracy WITH leakage:", leak_accuracy)

Accuracy WITH leakage: 1.0


In [10]:
X_clean = X_leak.drop(columns=["future_label"])

X_train,X_test,y_train,y_test=train_test_split(
X_clean,
y,
test_size=0.2,
random_state=42
)

model=RandomForestClassifier(
n_estimators=100,
random_state=42
)

model.fit(X_train,y_train)

pred=model.predict(X_test)

clean_accuracy=accuracy_score(y_test,pred)

print("Accuracy AFTER removing leakage:",clean_accuracy)

Accuracy AFTER removing leakage: 0.6494


LIMITATION

This notebook uses only March 2026 data and
does not evaluate temporal generalization across
multiple months.